---
# Lab 1

In [1]:
!apt-get update -qq
!pip install -q pyspark
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.12/dist-packages/pyspark"
os.environ["PATH"] += ":/usr/lib/jvm/java-17-openjdk-amd64/bin"

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PySpark Student Lab") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"Ready! Spark {spark.version}")

Ready! Spark 4.0.2


## Section 1 — Practice Exercises

In [4]:
students = spark.createDataFrame([
    (101, "Ali",     "CS",  88),
    (102, "Mona",    "CS",  95),
    (103, "Youssef", "IT",  72),
    (104, "Nour",    "IT",  81),
    (105, "Sara",    "AI",  91),
    (106, "Omar",    "AI",  85)
], ["student_id", "name", "major", "score"])
students.show()

+----------+-------+-----+-----+
|student_id|   name|major|score|
+----------+-------+-----+-----+
|       101|    Ali|   CS|   88|
|       102|   Mona|   CS|   95|
|       103|Youssef|   IT|   72|
|       104|   Nour|   IT|   81|
|       105|   Sara|   AI|   91|
|       106|   Omar|   AI|   85|
+----------+-------+-----+-----+



### Exercise 1 — Filter AI students with score > 90

In [5]:
result = students.filter((F.col("major") == "AI") & (F.col("score") > 90))
result.show()

+----------+----+-----+-----+
|student_id|name|major|score|
+----------+----+-----+-----+
|       105|Sara|   AI|   91|
+----------+----+-----+-----+



### Exercise 2 — Add `result` column (Pass / Fail)

In [6]:
result = students.withColumn(
    "result",
    F.when(F.col("score") >= 80, "Pass").otherwise("Fail")
)
result.show()

+----------+-------+-----+-----+------+
|student_id|   name|major|score|result|
+----------+-------+-----+-----+------+
|       101|    Ali|   CS|   88|  Pass|
|       102|   Mona|   CS|   95|  Pass|
|       103|Youssef|   IT|   72|  Fail|
|       104|   Nour|   IT|   81|  Pass|
|       105|   Sara|   AI|   91|  Pass|
|       106|   Omar|   AI|   85|  Pass|
+----------+-------+-----+-----+------+



### Exercise 3 — Average score per major (sorted highest → lowest)

In [7]:
result = (students
    .groupBy("major")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
result.show()

+-----+---------+
|major|avg_score|
+-----+---------+
|   CS|     91.5|
|   AI|     88.0|
|   IT|     76.5|
+-----+---------+



### Exercise 4 — Count distinct majors

In [8]:
result = students.select(F.countDistinct("major").alias("distinct_majors"))
result.show()

+---------------+
|distinct_majors|
+---------------+
|              3|
+---------------+



### Exercise 5 — Rename `major` → `department`, drop `student_id`

In [9]:
result = students.withColumnRenamed("major", "department").drop("student_id")
result.show()

+-------+----------+-----+
|   name|department|score|
+-------+----------+-----+
|    Ali|        CS|   88|
|   Mona|        CS|   95|
|Youssef|        IT|   72|
|   Nour|        IT|   81|
|   Sara|        AI|   91|
|   Omar|        AI|   85|
+-------+----------+-----+



## Section 2 — Window Function Exercises

### Exercise 1 — Running total of scores ordered by student_id

In [10]:
w = Window.orderBy("student_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = students.withColumn("running_total", F.sum("score").over(w))
result.show()

+----------+-------+-----+-----+-------------+
|student_id|   name|major|score|running_total|
+----------+-------+-----+-----+-------------+
|       101|    Ali|   CS|   88|           88|
|       102|   Mona|   CS|   95|          183|
|       103|Youssef|   IT|   72|          255|
|       104|   Nour|   IT|   81|          336|
|       105|   Sara|   AI|   91|          427|
|       106|   Omar|   AI|   85|          512|
+----------+-------+-----+-----+-------------+



### Exercise 2 — above_major_avg / below_major_avg

In [11]:
w_major = Window.partitionBy("major")
result = (students
    .withColumn("major_avg", F.avg("score").over(w_major))
    .withColumn("vs_major_avg",
        F.when(F.col("score") >= F.col("major_avg"), "above_major_avg")
         .otherwise("below_major_avg"))
    .drop("major_avg")
)
result.show()

+----------+-------+-----+-----+---------------+
|student_id|   name|major|score|   vs_major_avg|
+----------+-------+-----+-----+---------------+
|       105|   Sara|   AI|   91|above_major_avg|
|       106|   Omar|   AI|   85|below_major_avg|
|       101|    Ali|   CS|   88|below_major_avg|
|       102|   Mona|   CS|   95|above_major_avg|
|       103|Youssef|   IT|   72|below_major_avg|
|       104|   Nour|   IT|   81|above_major_avg|
+----------+-------+-----+-----+---------------+



### Exercise 3 — Top 2 students per major

In [12]:
w_rank = Window.partitionBy("major").orderBy(F.col("score").desc())
result = (students
    .withColumn("rank", F.rank().over(w_rank))
    .filter(F.col("rank") <= 2)
)
result.show()

+----------+-------+-----+-----+----+
|student_id|   name|major|score|rank|
+----------+-------+-----+-----+----+
|       105|   Sara|   AI|   91|   1|
|       106|   Omar|   AI|   85|   2|
|       102|   Mona|   CS|   95|   1|
|       101|    Ali|   CS|   88|   2|
|       104|   Nour|   IT|   81|   1|
|       103|Youssef|   IT|   72|   2|
+----------+-------+-----+-----+----+



## Section 3 — String Function Exercises

In [13]:
contacts = spark.createDataFrame([
    ("  ali123@example.com ",),
    ("sara_99@gmail.com",),
    ("mona.test@yahoo.com",),
    (" ahmed2025@outlook.com ",)
], ["email"])
contacts.show(truncate=False)

+-----------------------+
|email                  |
+-----------------------+
|  ali123@example.com   |
|sara_99@gmail.com      |
|mona.test@yahoo.com    |
| ahmed2025@outlook.com |
+-----------------------+



### Exercise 1 — Extract username and email provider

In [14]:
result = (contacts
    .withColumn("email",    F.trim(F.lower(F.col("email"))))
    .withColumn("username", F.regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("provider", F.regexp_extract("email", r'@([^.]+)', 1))
)
result.show(truncate=False)

+---------------------+---------+--------+
|email                |username |provider|
+---------------------+---------+--------+
|ali123@example.com   |ali123   |example |
|sara_99@gmail.com    |sara_99  |gmail   |
|mona.test@yahoo.com  |mona.test|yahoo   |
|ahmed2025@outlook.com|ahmed2025|outlook |
+---------------------+---------+--------+



### Exercise 2 — Domain name only (gmail, yahoo, outlook …)

In [15]:
result = (contacts
    .withColumn("email",  F.trim(F.lower(F.col("email"))))
    .withColumn("domain", F.regexp_extract("email", r'@([^.]+)', 1))
)
result.show(truncate=False)

+---------------------+-------+
|email                |domain |
+---------------------+-------+
|ali123@example.com   |example|
|sara_99@gmail.com    |gmail  |
|mona.test@yahoo.com  |yahoo  |
|ahmed2025@outlook.com|outlook|
+---------------------+-------+



### Exercise 3 — Clean email (trim + lowercase)

In [16]:
result = contacts.withColumn("email", F.trim(F.lower(F.col("email"))))
result.show(truncate=False)

+---------------------+
|email                |
+---------------------+
|ali123@example.com   |
|sara_99@gmail.com    |
|mona.test@yahoo.com  |
|ahmed2025@outlook.com|
+---------------------+



### Exercise 4 — Count emails per provider

In [17]:
result = (contacts
    .withColumn("email",    F.trim(F.lower(F.col("email"))))
    .withColumn("provider", F.regexp_extract("email", r'@([^.]+)', 1))
    .groupBy("provider")
    .agg(F.count("*").alias("email_count"))
)
result.show()

+--------+-----------+
|provider|email_count|
+--------+-----------+
|   gmail|          1|
| example|          1|
| outlook|          1|
|   yahoo|          1|
+--------+-----------+



### Exercise 5 — Remove digits from usernames

In [18]:
result = (contacts
    .withColumn("email",             F.trim(F.lower(F.col("email"))))
    .withColumn("username_raw",      F.regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("username_no_digits",F.regexp_replace("username_raw", r'[0-9]', ""))
)
result.select("email", "username_raw", "username_no_digits").show(truncate=False)

+---------------------+------------+------------------+
|email                |username_raw|username_no_digits|
+---------------------+------------+------------------+
|ali123@example.com   |ali123      |ali               |
|sara_99@gmail.com    |sara_99     |sara_             |
|mona.test@yahoo.com  |mona.test   |mona.test         |
|ahmed2025@outlook.com|ahmed2025   |ahmed             |
+---------------------+------------+------------------+



## Section 4 — Date & Time Exercises

In [19]:
events = spark.createDataFrame([
    (1, "2025-01-15"),
    (2, "2025-03-22"),
    (3, "2025-07-04"),
    (4, "2025-10-19"),
    (5, "2025-12-31")
], ["event_id", "event_date"])
events.show()

+--------+----------+
|event_id|event_date|
+--------+----------+
|       1|2025-01-15|
|       2|2025-03-22|
|       3|2025-07-04|
|       4|2025-10-19|
|       5|2025-12-31|
+--------+----------+



### Exercise 1 — Convert event_date to DateType

In [20]:
events = events.withColumn("event_date", F.to_date("event_date", "yyyy-MM-dd"))
events.printSchema()
events.show()

root
 |-- event_id: long (nullable = true)
 |-- event_date: date (nullable = true)

+--------+----------+
|event_id|event_date|
+--------+----------+
|       1|2025-01-15|
|       2|2025-03-22|
|       3|2025-07-04|
|       4|2025-10-19|
|       5|2025-12-31|
+--------+----------+



### Exercise 2 — Extract year, month, day

In [21]:
result = (events
    .withColumn("year",  F.year("event_date"))
    .withColumn("month", F.month("event_date"))
    .withColumn("day",   F.dayofmonth("event_date"))
)
result.show()

+--------+----------+----+-----+---+
|event_id|event_date|year|month|day|
+--------+----------+----+-----+---+
|       1|2025-01-15|2025|    1| 15|
|       2|2025-03-22|2025|    3| 22|
|       3|2025-07-04|2025|    7|  4|
|       4|2025-10-19|2025|   10| 19|
|       5|2025-12-31|2025|   12| 31|
+--------+----------+----+-----+---+



### Exercise 3 — Quarter of each event

In [22]:
result = events.withColumn(
    "quarter",
    F.when(F.month("event_date").between(1, 3),  "Q1")
     .when(F.month("event_date").between(4, 6),  "Q2")
     .when(F.month("event_date").between(7, 9),  "Q3")
     .otherwise("Q4")
)
result.show()

+--------+----------+-------+
|event_id|event_date|quarter|
+--------+----------+-------+
|       1|2025-01-15|     Q1|
|       2|2025-03-22|     Q1|
|       3|2025-07-04|     Q3|
|       4|2025-10-19|     Q4|
|       5|2025-12-31|     Q4|
+--------+----------+-------+



### Exercise 4 — Days remaining until end of event's year

In [23]:
result = events.withColumn(
    "days_until_year_end",
    F.datediff(
        F.to_date(F.concat(F.year("event_date").cast("string"), F.lit("-12-31")), "yyyy-MM-dd"),
        F.col("event_date")
    )
)
result.show()

+--------+----------+-------------------+
|event_id|event_date|days_until_year_end|
+--------+----------+-------------------+
|       1|2025-01-15|                350|
|       2|2025-03-22|                284|
|       3|2025-07-04|                180|
|       4|2025-10-19|                 73|
|       5|2025-12-31|                  0|
+--------+----------+-------------------+



### Exercise 5 — Events in the second half of the year (July–December)

In [24]:
result = events.filter(F.month("event_date") >= 7)
result.show()

+--------+----------+
|event_id|event_date|
+--------+----------+
|       3|2025-07-04|
|       4|2025-10-19|
|       5|2025-12-31|
+--------+----------+



## Section 5 — Array Function Exercises

In [25]:
courses = spark.createDataFrame([
    ("Python",  ["Programming", "Beginner", "Data"]),
    ("Spark",   ["Big Data", "Programming"]),
    ("SQL",     ["Database", "Data"]),
    ("Airflow", ["Workflow", "Automation", "Data"]),
    ("Linux",   ["OS", "Administration"])
], ["course_name", "topics"])
courses.show(truncate=False)

+-----------+-----------------------------+
|course_name|topics                       |
+-----------+-----------------------------+
|Python     |[Programming, Beginner, Data]|
|Spark      |[Big Data, Programming]      |
|SQL        |[Database, Data]             |
|Airflow    |[Workflow, Automation, Data] |
|Linux      |[OS, Administration]         |
+-----------+-----------------------------+



### Exercise 1 — Courses with more than 2 topics

In [26]:
result = courses.filter(F.size("topics") > 2)
result.show(truncate=False)

+-----------+-----------------------------+
|course_name|topics                       |
+-----------+-----------------------------+
|Python     |[Programming, Beginner, Data]|
|Airflow    |[Workflow, Automation, Data] |
+-----------+-----------------------------+



### Exercise 2 — has_data_topic column (True/False)

In [27]:
result = courses.withColumn("has_data_topic", F.array_contains("topics", "Data"))
result.show(truncate=False)

+-----------+-----------------------------+--------------+
|course_name|topics                       |has_data_topic|
+-----------+-----------------------------+--------------+
|Python     |[Programming, Beginner, Data]|true          |
|Spark      |[Big Data, Programming]      |false         |
|SQL        |[Database, Data]             |true          |
|Airflow    |[Workflow, Automation, Data] |true          |
|Linux      |[OS, Administration]         |false         |
+-----------+-----------------------------+--------------+



### Exercise 3 — Count courses per topic using explode()

In [28]:
result = (courses
    .withColumn("topic", F.explode("topics"))
    .groupBy("topic")
    .agg(F.count("course_name").alias("course_count"))
    .orderBy("course_count", ascending=False)
)
result.show()

+--------------+------------+
|         topic|course_count|
+--------------+------------+
|          Data|           3|
|   Programming|           2|
|      Big Data|           1|
|      Beginner|           1|
|Administration|           1|
|      Workflow|           1|
|            OS|           1|
|    Automation|           1|
|      Database|           1|
+--------------+------------+



### Exercise 4 — Courses containing the topic "Programming"

In [29]:
result = courses.filter(F.array_contains("topics", "Programming"))
result.show(truncate=False)

+-----------+-----------------------------+
|course_name|topics                       |
+-----------+-----------------------------+
|Python     |[Programming, Beginner, Data]|
|Spark      |[Big Data, Programming]      |
+-----------+-----------------------------+



### Exercise 5 — Number of topics per course

In [30]:
result = courses.withColumn("topic_count", F.size("topics"))
result.show(truncate=False)

+-----------+-----------------------------+-----------+
|course_name|topics                       |topic_count|
+-----------+-----------------------------+-----------+
|Python     |[Programming, Beginner, Data]|3          |
|Spark      |[Big Data, Programming]      |2          |
|SQL        |[Database, Data]             |2          |
|Airflow    |[Workflow, Automation, Data] |3          |
|Linux      |[OS, Administration]         |2          |
+-----------+-----------------------------+-----------+



## Capstone — Online Learning Platform Analytics

In [31]:
students_cap = spark.createDataFrame([
    (1, "Ali"), (2, "Mona"), (3, "Sara"), (4, "Omar"), (5, "Nour")
], ["student_id", "student_name"])

enrollments = spark.createDataFrame([
    (1, "Python", 88), (1, "SQL", 92),
    (2, "Python", 95), (2, "Spark", 90),
    (3, "SQL",    78), (3, "Spark", 85),
    (4, "Python", 70),
    (5, "Spark",  98)
], ["student_id", "course", "score"])

print("students_cap:"); students_cap.show()
print("enrollments:");  enrollments.show()

students_cap:
+----------+------------+
|student_id|student_name|
+----------+------------+
|         1|         Ali|
|         2|        Mona|
|         3|        Sara|
|         4|        Omar|
|         5|        Nour|
+----------+------------+

enrollments:
+----------+------+-----+
|student_id|course|score|
+----------+------+-----+
|         1|Python|   88|
|         1|   SQL|   92|
|         2|Python|   95|
|         2| Spark|   90|
|         3|   SQL|   78|
|         3| Spark|   85|
|         4|Python|   70|
|         5| Spark|   98|
+----------+------+-----+



### Part 1 — Data Preparation
#### Task 1 — Join students with enrollments

In [32]:
joined = students_cap.join(enrollments, on="student_id", how="inner")
joined.show()

+----------+------------+------+-----+
|student_id|student_name|course|score|
+----------+------------+------+-----+
|         1|         Ali|Python|   88|
|         1|         Ali|   SQL|   92|
|         2|        Mona|Python|   95|
|         2|        Mona| Spark|   90|
|         3|        Sara|   SQL|   78|
|         3|        Sara| Spark|   85|
|         4|        Omar|Python|   70|
|         5|        Nour| Spark|   98|
+----------+------------+------+-----+



#### Task 2 — Schema and data quality

In [33]:
joined.printSchema()
print("Row count:", joined.count())
print("Null counts:")
joined.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in joined.columns]).show()

root
 |-- student_id: long (nullable = true)
 |-- student_name: string (nullable = true)
 |-- course: string (nullable = true)
 |-- score: long (nullable = true)

Row count: 8
Null counts:
+----------+------------+------+-----+
|student_id|student_name|course|score|
+----------+------------+------+-----+
|         0|           0|     0|    0|
+----------+------------+------+-----+



#### Task 3 — Remove duplicate records

In [34]:
joined = joined.dropDuplicates()
print("Row count after dedup:", joined.count())

Row count after dedup: 8


### Part 2 — Aggregations
#### Task 4 — Average score per student

In [35]:
avg_per_student = (joined
    .groupBy("student_id", "student_name")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
avg_per_student.show()

+----------+------------+---------+
|student_id|student_name|avg_score|
+----------+------------+---------+
|         5|        Nour|     98.0|
|         2|        Mona|     92.5|
|         1|         Ali|     90.0|
|         3|        Sara|     81.5|
|         4|        Omar|     70.0|
+----------+------------+---------+



#### Task 5 — Average score per course

In [36]:
avg_per_course = (joined
    .groupBy("course")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
avg_per_course.show()

+------+-----------------+
|course|        avg_score|
+------+-----------------+
| Spark|             91.0|
|   SQL|             85.0|
|Python|84.33333333333333|
+------+-----------------+



#### Task 6 — Highest score per course

In [37]:
highest_per_course = (joined
    .groupBy("course")
    .agg(F.max("score").alias("highest_score"))
)
highest_per_course.show()

+------+-------------+
|course|highest_score|
+------+-------------+
|   SQL|           92|
|Python|           95|
| Spark|           98|
+------+-------------+



#### Task 7 — Student with the highest average score overall

In [38]:
result = (joined
    .groupBy("student_id", "student_name")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
    .limit(1)
)
result.show()

+----------+------------+---------+
|student_id|student_name|avg_score|
+----------+------------+---------+
|         5|        Nour|     98.0|
+----------+------------+---------+



### Part 3 — Window Functions
#### Task 8 — Rank students within each course

In [39]:
w_course = Window.partitionBy("course").orderBy(F.col("score").desc())
ranked = joined.withColumn("course_rank", F.rank().over(w_course))
ranked.show()

+----------+------------+------+-----+-----------+
|student_id|student_name|course|score|course_rank|
+----------+------------+------+-----+-----------+
|         2|        Mona|Python|   95|          1|
|         1|         Ali|Python|   88|          2|
|         4|        Omar|Python|   70|          3|
|         1|         Ali|   SQL|   92|          1|
|         3|        Sara|   SQL|   78|          2|
|         5|        Nour| Spark|   98|          1|
|         2|        Mona| Spark|   90|          2|
|         3|        Sara| Spark|   85|          3|
+----------+------------+------+-----+-----------+



#### Task 9 — Running total of scores within each course

In [40]:
w_running = Window.partitionBy("course").orderBy("student_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = joined.withColumn("running_total", F.sum("score").over(w_running))
result.show()

+----------+------------+------+-----+-------------+
|student_id|student_name|course|score|running_total|
+----------+------------+------+-----+-------------+
|         1|         Ali|Python|   88|           88|
|         2|        Mona|Python|   95|          183|
|         4|        Omar|Python|   70|          253|
|         1|         Ali|   SQL|   92|           92|
|         3|        Sara|   SQL|   78|          170|
|         2|        Mona| Spark|   90|           90|
|         3|        Sara| Spark|   85|          175|
|         5|        Nour| Spark|   98|          273|
+----------+------------+------+-----+-------------+



#### Task 10 — Difference between student score and course average

In [41]:
w_avg = Window.partitionBy("course")
result = (joined
    .withColumn("course_avg", F.avg("score").over(w_avg))
    .withColumn("diff_from_avg", F.col("score") - F.col("course_avg"))
)
result.show()

+----------+------------+------+-----+-----------------+-------------------+
|student_id|student_name|course|score|       course_avg|      diff_from_avg|
+----------+------------+------+-----+-----------------+-------------------+
|         1|         Ali|Python|   88|84.33333333333333| 3.6666666666666714|
|         2|        Mona|Python|   95|84.33333333333333| 10.666666666666671|
|         4|        Omar|Python|   70|84.33333333333333|-14.333333333333329|
|         1|         Ali|   SQL|   92|             85.0|                7.0|
|         3|        Sara|   SQL|   78|             85.0|               -7.0|
|         2|        Mona| Spark|   90|             91.0|               -1.0|
|         3|        Sara| Spark|   85|             91.0|               -6.0|
|         5|        Nour| Spark|   98|             91.0|                7.0|
+----------+------------+------+-----+-----------------+-------------------+



### Part 4 — Business Rules
#### Task 11 — Performance category

In [42]:
with_perf = joined.withColumn(
    "performance",
    F.when(F.col("score") >= 90, "Excellent")
     .when(F.col("score") >= 80, "Good")
     .otherwise("Needs Improvement")
)
with_perf.show()

+----------+------------+------+-----+-----------------+
|student_id|student_name|course|score|      performance|
+----------+------------+------+-----+-----------------+
|         1|         Ali|Python|   88|             Good|
|         1|         Ali|   SQL|   92|        Excellent|
|         2|        Mona|Python|   95|        Excellent|
|         2|        Mona| Spark|   90|        Excellent|
|         3|        Sara|   SQL|   78|Needs Improvement|
|         3|        Sara| Spark|   85|             Good|
|         4|        Omar|Python|   70|Needs Improvement|
|         5|        Nour| Spark|   98|        Excellent|
+----------+------------+------+-----+-----------------+



#### Task 12 — Count students per performance category

In [43]:
result = (with_perf
    .groupBy("performance")
    .agg(F.count("student_id").alias("student_count"))
    .orderBy("student_count", ascending=False)
)
result.show()

+-----------------+-------------+
|      performance|student_count|
+-----------------+-------------+
|        Excellent|            4|
|Needs Improvement|            2|
|             Good|            2|
+-----------------+-------------+



### Part 5 — Final Report

In [44]:
w_final = Window.partitionBy("course").orderBy(F.col("score").desc())

final_report = (joined
    .withColumn("performance",
        F.when(F.col("score") >= 90, "Excellent")
         .when(F.col("score") >= 80, "Good")
         .otherwise("Needs Improvement"))
    .withColumn("course_rank", F.rank().over(w_final))
    .select("student_name", "course", "score", "course_rank", "performance")
    .orderBy("course", "course_rank")
)
final_report.show()

+------------+------+-----+-----------+-----------------+
|student_name|course|score|course_rank|      performance|
+------------+------+-----+-----------+-----------------+
|        Mona|Python|   95|          1|        Excellent|
|         Ali|Python|   88|          2|             Good|
|        Omar|Python|   70|          3|Needs Improvement|
|         Ali|   SQL|   92|          1|        Excellent|
|        Sara|   SQL|   78|          2|Needs Improvement|
|        Nour| Spark|   98|          1|        Excellent|
|        Mona| Spark|   90|          2|        Excellent|
|        Sara| Spark|   85|          3|             Good|
+------------+------+-----+-----------+-----------------+



---
# Lab 2

## Section 1 — Practice Exercises

In [45]:
employees = spark.createDataFrame([
    (1, "Anna",  "Engineering", 95000),
    (2, "Ben",   "Marketing",   72000),
    (3, "Clara", "Engineering", 88000),
    (4, "Dan",   "HR",          65000),
    (5, "Eva",   "Marketing",   79000),
    (6, "Frank", "Engineering", 91000),
], ["id", "name", "dept", "salary"])
employees.show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  1| Anna|Engineering| 95000|
|  2|  Ben|  Marketing| 72000|
|  3|Clara|Engineering| 88000|
|  4|  Dan|         HR| 65000|
|  5|  Eva|  Marketing| 79000|
|  6|Frank|Engineering| 91000|
+---+-----+-----------+------+



### Exercise 1 — Engineering employees with salary > 90,000

In [46]:
result = employees.filter((F.col("dept") == "Engineering") & (F.col("salary") > 90000))
result.show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  1| Anna|Engineering| 95000|
|  6|Frank|Engineering| 91000|
+---+-----+-----------+------+



### Exercise 2 — salary_grade column (High / Standard)

In [47]:
result = employees.withColumn(
    "salary_grade",
    F.when(F.col("salary") >= 80000, "High").otherwise("Standard")
)
result.show()

+---+-----+-----------+------+------------+
| id| name|       dept|salary|salary_grade|
+---+-----+-----------+------+------------+
|  1| Anna|Engineering| 95000|        High|
|  2|  Ben|  Marketing| 72000|    Standard|
|  3|Clara|Engineering| 88000|        High|
|  4|  Dan|         HR| 65000|    Standard|
|  5|  Eva|  Marketing| 79000|    Standard|
|  6|Frank|Engineering| 91000|        High|
+---+-----+-----------+------+------------+



### Exercise 3 — Average salary per department (sorted descending)

In [48]:
result = (employees
    .groupBy("dept")
    .agg(F.avg("salary").alias("avg_salary"))
    .orderBy("avg_salary", ascending=False)
)
result.show()

+-----------+-----------------+
|       dept|       avg_salary|
+-----------+-----------------+
|Engineering|91333.33333333333|
|  Marketing|          75500.0|
|         HR|          65000.0|
+-----------+-----------------+



### Exercise 4 — Count distinct departments

In [49]:
result = employees.select(F.countDistinct("dept").alias("distinct_depts"))
result.show()

+--------------+
|distinct_depts|
+--------------+
|             3|
+--------------+



### Exercise 5 — Rename `dept` → `department`, drop `id`

In [50]:
result = employees.withColumnRenamed("dept", "department").drop("id")
result.show()

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
| Anna|Engineering| 95000|
|  Ben|  Marketing| 72000|
|Clara|Engineering| 88000|
|  Dan|         HR| 65000|
|  Eva|  Marketing| 79000|
|Frank|Engineering| 91000|
+-----+-----------+------+



## Section 2 — Window Function Exercises

### Exercise 1 — Running total of salary ordered by id

In [51]:
w = Window.orderBy("id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = employees.withColumn("running_salary_total", F.sum("salary").over(w))
result.show()

+---+-----+-----------+------+--------------------+
| id| name|       dept|salary|running_salary_total|
+---+-----+-----------+------+--------------------+
|  1| Anna|Engineering| 95000|               95000|
|  2|  Ben|  Marketing| 72000|              167000|
|  3|Clara|Engineering| 88000|              255000|
|  4|  Dan|         HR| 65000|              320000|
|  5|  Eva|  Marketing| 79000|              399000|
|  6|Frank|Engineering| 91000|              490000|
+---+-----+-----------+------+--------------------+



### Exercise 2 — above_avg / below_avg per department

In [52]:
w_dept = Window.partitionBy("dept")
result = (employees
    .withColumn("dept_avg", F.avg("salary").over(w_dept))
    .withColumn("vs_dept_avg",
        F.when(F.col("salary") >= F.col("dept_avg"), "above_avg")
         .otherwise("below_avg"))
    .drop("dept_avg")
)
result.show()

+---+-----+-----------+------+-----------+
| id| name|       dept|salary|vs_dept_avg|
+---+-----+-----------+------+-----------+
|  1| Anna|Engineering| 95000|  above_avg|
|  3|Clara|Engineering| 88000|  below_avg|
|  6|Frank|Engineering| 91000|  below_avg|
|  4|  Dan|         HR| 65000|  above_avg|
|  2|  Ben|  Marketing| 72000|  below_avg|
|  5|  Eva|  Marketing| 79000|  above_avg|
+---+-----+-----------+------+-----------+



### Exercise 3 — Top 3 Engineering employees by salary (window function)

In [53]:
w_eng = Window.partitionBy("dept").orderBy(F.col("salary").desc())
result = (employees
    .withColumn("rank", F.rank().over(w_eng))
    .filter((F.col("dept") == "Engineering") & (F.col("rank") <= 3))
)
result.show()

+---+-----+-----------+------+----+
| id| name|       dept|salary|rank|
+---+-----+-----------+------+----+
|  1| Anna|Engineering| 95000|   1|
|  6|Frank|Engineering| 91000|   2|
|  3|Clara|Engineering| 88000|   3|
+---+-----+-----------+------+----+



## Section 3 — String Function Exercises

In [54]:
logs = spark.createDataFrame([
    ("2025-06-01 08:15:32 ERROR disk full",),
    ("2025-06-01 09:00:00 INFO  service started",),
    ("2025-06-02 14:45:11 WARN  memory low",),
    ("2025-06-03 23:59:59 ERROR connection timeout",),
], ["log_entry"])

emails_df = spark.createDataFrame([
    ("alice@gmail.com",),
    ("bob.smith@company.org",),
    ("clara99@yahoo.com",),
], ["email"])

names_df = spark.createDataFrame([
    ("  john doe 1st  ",),
    (" JANE  SMITH2  ",),
    ("  ali HASSAN  ",),
], ["name"])

### Exercise 1 — Extract date and log level from log entries

In [55]:
result = (logs
    .withColumn("log_date",  F.regexp_extract("log_entry", r'^(\d{4}-\d{2}-\d{2})', 1))
    .withColumn("log_level", F.trim(F.regexp_extract("log_entry", r'\d{2}:\d{2}:\d{2}\s+(\S+)', 1)))
)
result.show(truncate=False)

+--------------------------------------------+----------+---------+
|log_entry                                   |log_date  |log_level|
+--------------------------------------------+----------+---------+
|2025-06-01 08:15:32 ERROR disk full         |2025-06-01|ERROR    |
|2025-06-01 09:00:00 INFO  service started   |2025-06-01|INFO     |
|2025-06-02 14:45:11 WARN  memory low        |2025-06-02|WARN     |
|2025-06-03 23:59:59 ERROR connection timeout|2025-06-03|ERROR    |
+--------------------------------------------+----------+---------+



### Exercise 2 — Extract username and domain from email

In [56]:
result = (emails_df
    .withColumn("username", F.regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("domain",   F.regexp_extract("email", r'@(.+)$', 1))
)
result.show(truncate=False)

+---------------------+---------+-----------+
|email                |username |domain     |
+---------------------+---------+-----------+
|alice@gmail.com      |alice    |gmail.com  |
|bob.smith@company.org|bob.smith|company.org|
|clara99@yahoo.com    |clara99  |yahoo.com  |
+---------------------+---------+-----------+



### Exercise 3 — Clean name: trim, proper case, remove digits

In [57]:
result = (names_df
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("name", F.initcap(F.col("name")))
    .withColumn("name", F.regexp_replace("name", r'[0-9]', ""))
    .withColumn("name", F.trim(F.col("name")))
)
result.show(truncate=False)

+-----------+
|name       |
+-----------+
|John Doe st|
|Jane  Smith|
|Ali Hassan |
+-----------+



## Section 4 — Date & Time Exercises

In [58]:
orders = spark.createDataFrame([
    (1, "2025-01-05"),
    (2, "2025-04-12"),
    (3, "2025-06-21"),
    (4, "2025-09-07"),
    (5, "2025-11-29"),
], ["order_id", "order_date"])
orders = orders.withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
orders.show()

+--------+----------+
|order_id|order_date|
+--------+----------+
|       1|2025-01-05|
|       2|2025-04-12|
|       3|2025-06-21|
|       4|2025-09-07|
|       5|2025-11-29|
+--------+----------+



### Exercise 1 — Order age in days, quarter, last day of month

In [59]:
result = (orders
    .withColumn("age_days",   F.datediff(F.current_date(), "order_date"))
    .withColumn("quarter",
        F.when(F.month("order_date").between(1, 3),  "Q1")
         .when(F.month("order_date").between(4, 6),  "Q2")
         .when(F.month("order_date").between(7, 9),  "Q3")
         .otherwise("Q4"))
    .withColumn("last_day_of_month", F.last_day("order_date"))
)
result.show()

+--------+----------+--------+-------+-----------------+
|order_id|order_date|age_days|quarter|last_day_of_month|
+--------+----------+--------+-------+-----------------+
|       1|2025-01-05|     512|     Q1|       2025-01-31|
|       2|2025-04-12|     415|     Q2|       2025-04-30|
|       3|2025-06-21|     345|     Q2|       2025-06-30|
|       4|2025-09-07|     267|     Q3|       2025-09-30|
|       5|2025-11-29|     184|     Q4|       2025-11-30|
+--------+----------+--------+-------+-----------------+



### Exercise 2 — Orders placed on a weekend (Saturday or Sunday)

In [60]:
# dayofweek: 1=Sunday, 7=Saturday
result = orders.filter(F.dayofweek("order_date").isin(1, 7))
result.show()

+--------+----------+
|order_id|order_date|
+--------+----------+
|       1|2025-01-05|
|       2|2025-04-12|
|       3|2025-06-21|
|       4|2025-09-07|
|       5|2025-11-29|
+--------+----------+



## Section 5 — Array Function Exercises

In [61]:
products = spark.createDataFrame([
    (1, "Laptop",  ["electronics", "computers", "work"]),
    (2, "T-Shirt", ["clothing", "fashion"]),
    (3, "Phone",   ["electronics", "mobile"]),
    (4, "Headset", ["electronics", "audio", "computers"]),
    (5, "Jacket",  ["clothing", "fashion", "sale"]),
], ["id", "product", "tags"])

promo_products = spark.createDataFrame([
    (1, ["sale", "featured"]),
    (2, ["new",  "sale"]),
    (3, ["featured"]),
    (4, ["sale", "electronics"]),
    (5, ["clearance"]),
], ["id", "promo_tags"])

### Exercise 1 — Products with more than 2 tags

In [62]:
result = products.filter(F.size("tags") > 2)
result.show(truncate=False)

+---+-------+-------------------------------+
|id |product|tags                           |
+---+-------+-------------------------------+
|1  |Laptop |[electronics, computers, work] |
|4  |Headset|[electronics, audio, computers]|
|5  |Jacket |[clothing, fashion, sale]      |
+---+-------+-------------------------------+



### Exercise 2 — has_sale_tag column from promo_tags

In [63]:
result = promo_products.withColumn("has_sale_tag", F.array_contains("promo_tags", "sale"))
result.show(truncate=False)

+---+-------------------+------------+
|id |promo_tags         |has_sale_tag|
+---+-------------------+------------+
|1  |[sale, featured]   |true        |
|2  |[new, sale]        |true        |
|3  |[featured]         |false       |
|4  |[sale, electronics]|true        |
|5  |[clearance]        |false       |
+---+-------------------+------------+



### Exercise 3 — Count products per tag using explode() + groupBy()

In [64]:
result = (products
    .withColumn("tag", F.explode("tags"))
    .groupBy("tag")
    .agg(F.count("id").alias("product_count"))
    .orderBy("product_count", ascending=False)
)
result.show()

+-----------+-------------+
|        tag|product_count|
+-----------+-------------+
|electronics|            3|
|  computers|            2|
|   clothing|            2|
|    fashion|            2|
|       work|            1|
|     mobile|            1|
|      audio|            1|
|       sale|            1|
+-----------+-------------+

